# Which change made RF-DETR's solve slower? (revision bisect on one GPU)

The release candidate (`develop`) solves RF-DETR's shapes 1.02 to 1.22 times slower than the shipped revision `881e5c3` in a same-session A/B on an L4, and the larger the number of targets and the batch, the larger the gap. Four merged changes touch the solver between the two. This notebook runs the repository's `rfdetr` benchmark preset at each of them, on this GPU in this session, so the host CPU and the GPU are the same for every revision, and prints the warm median per shape relative to the shipped revision.

| Revision | Change merged there |
| --- | --- |
| `881e5c3` | shipped baseline |
| `dc6b06f` | solve costs on their own device, 64-bit offsets |
| `6f24fe8` | matrix dimensions passed at runtime, in-kernel row reduction |
| `9c9c43a` | indices API, validation mode, small-batch routing |
| `19477d9` | four warps for large shapes (RF-DETR shapes keep one warp) |

The first revision with a clearly larger ratio is the change that costs time. The baseline is run a second time at the end: the difference between its two runs is the noise floor of this session, and a gap smaller than that is not evidence.

In [ ]:
import torch

# The Triton kernel needs compute capability 8.0 or newer, so fail here instead of timing the SciPy fallback.
assert torch.cuda.is_available(), "Choose a GPU runtime: Runtime -> Change runtime type"
capability = torch.cuda.get_device_capability()
assert capability >= (8, 0), (
    f"{torch.cuda.get_device_name()} is sm_{capability[0]}{capability[1]}; the Triton backend needs sm_80 or newer"
)
print(torch.cuda.get_device_name(), f"sm_{capability[0]}{capability[1]}", "torch", torch.__version__)

In [ ]:
# Triton's version decides how the kernel compiles, so the result is only comparable between sessions with the same one.
import triton

print("triton", triton.__version__)

## 1. Get the repository

The benchmark script is checked out together with the sources at each revision, so every run uses its own revision's solver and its own benchmark options.

In [ ]:
import subprocess
from pathlib import Path

REPO = Path("/content/torch-hungarian")

if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/Borda/torch-hungarian.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin"], check=True)

In [ ]:
# One label per run, in this order. The last entry repeats the baseline to measure the noise of the session itself.
RUNS = [
    ("shipped", "881e5c3"),
    ("64-bit offsets", "dc6b06f"),
    ("runtime dims", "6f24fe8"),
    ("indices and routing", "9c9c43a"),
    ("four warps", "19477d9"),
    ("shipped again", "881e5c3"),
]

## 2. Benchmark each revision

Each run uses the `rfdetr` preset: 300 workers, 1 to 30 targets, batches of 52, 208 and 624, three fresh processes per case. A run whose output file already exists is skipped, so a restarted session continues where it stopped.

In [ ]:
import os


def run_rfdetr_preset(label: str, ref: str) -> Path:
    """Benchmark the `rfdetr` preset at one revision, importing the solver from that checkout."""
    output = Path("/content") / f"bisect-{label.replace(' ', '-')}-{ref}.jsonl"
    if output.exists():
        return output
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "--detach", ref], check=True)
    command = [
        "python",
        str(REPO / "tests/benchmark.py"),
        "--preset",
        "rfdetr",
        "--backends",
        "scipy,public_cuda,triton",
        "--dtypes",
        "float32",
        "--process-rounds",
        "3",
        "--output",
        str(output),
    ]
    # PYTHONPATH puts the checked-out sources first, so the run measures this revision and not an installed copy.
    subprocess.run(command, cwd=REPO, env={**os.environ, "PYTHONPATH": str(REPO / "src")}, check=True)
    return output

In [ ]:
outputs = {}
outputs[RUNS[0][0]] = run_rfdetr_preset(*RUNS[0])

In [ ]:
outputs[RUNS[1][0]] = run_rfdetr_preset(*RUNS[1])

In [ ]:
outputs[RUNS[2][0]] = run_rfdetr_preset(*RUNS[2])

In [ ]:
outputs[RUNS[3][0]] = run_rfdetr_preset(*RUNS[3])

In [ ]:
outputs[RUNS[4][0]] = run_rfdetr_preset(*RUNS[4])

In [ ]:
outputs[RUNS[5][0]] = run_rfdetr_preset(*RUNS[5])

## 3. Compare against the shipped revision

For each revision the table lists the warm median of the public call (the entry point RF-DETR reaches, default validation) per shape, divided by the first shipped run's median for the same shape. A ratio above 1 means slower than shipped.

In [ ]:
import json


def warm_medians(path: Path, backend: str, validation: str) -> dict[tuple[int, int], float]:
    """Return the median-over-rounds warm time in ms per (batch, targets) for one backend and validation mode."""
    medians = {}
    for line in path.read_text().splitlines():
        record = json.loads(line) if line.strip() else {}
        if (
            record.get("record_type") == "case"
            and record.get("status") == "measured"
            and record["backend"] == backend
            and record["validation_mode"] == validation
        ):
            shape = record["shape"]
            medians[(shape["batch"], shape["tasks"])] = record["process_statistics"]["warm_median_ms"]["median"]
    return medians

In [ ]:
import pandas as pd

public = {label: warm_medians(path, "public_cuda", "full") for label, path in outputs.items()}
reference = public["shipped"]
assert reference, "the shipped run produced no measured public-call rows"
assert all(run.keys() == reference.keys() for run in public.values()), "every revision must measure the same shapes"
ratios = pd.DataFrame(
    {label: {shape: run[shape] / reference[shape] for shape in sorted(reference)} for label, run in public.items()}
)
ratios.index = [f"B={batch} T={tasks}" for batch, tasks in ratios.index]
display(ratios.round(2))

In [ ]:
# One number per revision; the median over shapes is robust to a few noisy rows.
display(ratios.median().round(3).rename("median ratio to shipped").to_frame())

Read the second table from top to bottom. The step up between two neighbours is the cost of the change merged there, and `shipped again` against `shipped` is the noise floor: any step that is not clearly larger than that gap is not a finding. A step that appears with the runtime-dimension revision points at the kernel losing its per-shape constants; one that appears with the 64-bit offsets points at index arithmetic; one that appears with the indices and routing revision points at the new call path or validation.

In [ ]:
# The same comparison on the kernel with validation off, which removes the validation and conversion work from the picture.
private = {label: warm_medians(path, "triton", "off") for label, path in outputs.items()}
kernel = pd.DataFrame(
    {
        label: {shape: run[shape] / private["shipped"][shape] for shape in sorted(private["shipped"])}
        for label, run in private.items()
    }
)
display(kernel.median().round(3).rename("median ratio, validation off").to_frame())

If the ratios with validation off show the same steps, the cost is in the kernel itself. If only the public-call ratios step up, it is in the surrounding code. Post the two summary tables with the output of the first cells, which record the GPU, torch and Triton versions.